# Relevance judging (decision 0022)

Gemma 4 31B judges every (query, document) pair of a pool on the 0-3 scale with the fixed prompt in
`src/benchmark/judge_prompt.py` (UMBRELA-based; the changes are listed in that file). Greedy decoding stands in for
temperature 0. The judge sees only the query, the document's title and the first 800 words of its text.

Pairs of the author's pilot sheet are judged first, so the pilot agreement (`src.benchmark.judging agreement`) can be
checked before the rest of the pool. The run can be restarted: pairs already judged are skipped.

**Before running**
- Kaggle, accelerator GPU T4 x2, internet on. The model loads in 4-bit across both GPUs (about 18 GB of weights).
- Run the cells interactively until the first pilot pairs print. A T4 has no bfloat16, so the model computes in
  float16; if the printed outputs are empty, repeated or have no score line, stop and report it instead of changing
  settings. If they look right, use Save Version, Save & Run All, and close the tab.

**Output**: `relevance_judge_v1/<split>_judgements_v1.jsonl`, one line per pair with the label, the rationale, the raw
output, the model and revision, the prompt version and its SHA-256, settings, library versions and date. Place it at
`results/judging/<split>_judgements_v1.jsonl` in the repository.

In [ ]:
REPO = "https://github.com/azzs11/heritagerec.git"
BRANCH = "main"
SPLIT = "dev"
MODEL = "google/gemma-4-31B-it"  # Apache-2.0 (model card checked 22 Nov 2026)
REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"  # the revision recorded in decision 0022
GENERATION = {"do_sample": False, "max_new_tokens": 160}
OUT_DIR = "relevance_judge_v1"

In [ ]:
!git clone --depth 1 -b $BRANCH $REPO heritagerec
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import csv
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import bitsandbytes
import torch
import transformers
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

sys.path.insert(0, "heritagerec")
from src.benchmark.judge_prompt import PROMPT_VERSION, judge_prompt, parse, template_sha256

pairs = [json.loads(line) for line in open(f"heritagerec/data/judging/{SPLIT}_pool_v1.jsonl", encoding="utf-8")]
pilot_path = Path(f"heritagerec/data/judging/{SPLIT}_pilot_sheet_v1.csv")
# Only the pair ids are read from the pilot sheet; the judge never sees the author's labels.
pilot_ids = (
    {r["pair_id"] for r in csv.DictReader(pilot_path.open(encoding="utf-8-sig"))} if pilot_path.exists() else set()
)
out_path = Path(OUT_DIR) / f"{SPLIT}_judgements_v1.jsonl"
out_path.parent.mkdir(exist_ok=True)
done = {json.loads(line)["pair_id"] for line in out_path.open(encoding="utf-8")} if out_path.exists() else set()
todo = sorted(
    (p for p in pairs if p["pair_id"] not in done), key=lambda p: (p["pair_id"] not in pilot_ids, p["pair_id"])
)
environment = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "repo_commit": subprocess.run(
        ["git", "-C", "heritagerec", "rev-parse", "HEAD"], capture_output=True, text=True
    ).stdout.strip(),
}
print(len(pairs), "pairs;", len(pilot_ids), "pilot;", len(done), "already judged;", len(todo), "to do;", environment)

In [ ]:
compute = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=compute)
processor = AutoProcessor.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForMultimodalLM.from_pretrained(MODEL, revision=REVISION, quantization_config=quant, device_map="auto")
model.eval()


def judge(prompt: str) -> tuple[str, int]:
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    enc = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=GENERATION["max_new_tokens"], do_sample=GENERATION["do_sample"])
    n_in = enc["input_ids"].shape[1]
    return processor.decode(out[0, n_in:], skip_special_tokens=True).strip(), n_in

In [ ]:
for i, p in enumerate(todo, 1):
    prompt = judge_prompt(p["query"], p["title"], p["text"])
    raw, n_in = judge(prompt)
    label, rationale = parse(raw)
    record = {
        "pair_id": p["pair_id"],
        "qid": p["qid"],
        "doc_id": p["doc_id"],
        "label": label,
        "rationale": rationale,
        "raw": raw,
        "pilot": p["pair_id"] in pilot_ids,
        "input_tokens": n_in,
        "model": MODEL,
        "revision": REVISION,
        "quantization": f"4-bit nf4, compute {str(compute).replace('torch.', '')}",
        "prompt_version": PROMPT_VERSION,
        "prompt_sha256": template_sha256(),
        "generation": GENERATION,
        "environment": environment,
        "date": datetime.now(timezone.utc).isoformat(),
    }
    with out_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    if i <= 10 or i % 50 == 0:
        print(i, p["qid"], p["doc_id"], "label", label, "|", rationale[:160])
print(
    "judged",
    len(todo),
    "pairs; outputs without a score line:",
    sum(1 for line in out_path.open() if json.loads(line)["label"] is None),
)

In [ ]:
try:
    from google.colab import files

    files.download(str(out_path))
except ImportError:
    # On Kaggle the file is in the Output panel under relevance_judge_v1/.
    print("Download", out_path.resolve(), "from the notebook's output files")